# Remote LLM Inference: Running Modern Models on Google Colab

* * *

<div class="alert alert-success">  
    
### Learning Objectives
    
* Run modern language models on Google Colab
* Understand practical limits of remote inference
* Implement efficient processing for research tasks
</div>

**PLEASE MAKE SURE TO OPEN THIS NOTEBOOK IN GOOGLE COLAB: [colab.research.google.com](colab.research.google.com)**

### Models
*Updated 11/09/2025*

| Model | Size | RAM Needed | Notable Features |
|-------|------|------------|-----------------|
| Qwen3Guard-Gen-0.6B | 0.6B | 2GB | Latest safety-focused model, 119 languages |
| TinyLlama-1.1B | 1.1B | 3GB | Fastest, good for basic tasks |
| Qwen3-4B | 4B | 7GB | Latest Alibaba model, thinking/non-thinking modes |
| Phi-2 | 2.7B | 6GB | Microsoft, excellent reasoning |
| Yi-6B | 6B | 8GB | Strong bilingual, excellent for code |
| Neural-Chat-7B-v3-1 | 7B | 9GB | Optimized for chat, Intel |
| Qwen2.5-7B | 7B | 9GB | Stable Qwen model, strong general performance |
| GPT-OSS-20B | 20B | 16GB | Strong general performance |

All models are open-weights and available on Hugging Face. Models are listed from smallest to largest, with specialized models noted for their strengths. The 0.6B-4B models are particularly suitable for local inference on laptops.

### Sections
1. [Hardware Detection](#setup)
2. [Simple Installation](#install)
3. [Loading Models](#load)
4. [Text Generation](#generation)
5. [Practical Examples](#examples)
6. [Performance Tips](#performance)

<a id='setup'></a>

# Google Colab GPU Setup

**Colab GPU Tiers and Compatible Models:**

- **T4 GPU (Free Tier - 16GB)**: Qwen2.5-7B, Llama-3.2-3B, Phi-3.5-mini
- **L4 GPU (Colab Pro - 24GB)**: Qwen2.5-14B, Mistral-Nemo-12B
- **A100 GPU (Colab Pro+ - 40GB)**: Qwen2.5-32B, Mistral Small 3

## Enable and Auto-Detect GPU

**Step 1: Enable GPU Runtime**
1. Go to `Runtime` → `Change runtime type`
2. Set `Hardware accelerator` to `T4 GPU` (or L4/A100 if you have Pro/Pro+)
3. Switch on 'High-RAM' mode which allocates more memory.
4. Click `Save` (this will restart your runtime)

In [17]:
# Check Colab system resources
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

# Check RAM
!free -h

# Check disk space
!df -h /content

Tesla T4, 15360 MiB, 1740 MiB
               total        used        free      shared  buff/cache   available
Mem:            50Gi       9.5Gi       7.4Gi        16Mi        34Gi        40Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   54G  183G  23% /


<a id='install'></a>

# Installation

In [26]:
# Install only essential packages
!pip install -q torch transformers accelerate
!pip install -q psutil  # For system monitoring

# Verify versions
import transformers
import torch
print(f"\nTransformers version: {transformers.__version__}")
print(f"PyTorch version: {torch.__version__}")


Transformers version: 4.57.1
PyTorch version: 2.8.0+cu126


<a id='load'></a>

# Loading Models

Load a model. We'll use standard transformers library - simple and reliable!

An “Instruct model” (short for instruction-tuned model) is a large language model that has been fine-tuned to follow human instructions in natural language.

It starts from a base model (which just predicts the next token in raw text), and then goes through an extra supervised fintuning phase (*instruction tuning*) where the model learns to interpret instructions (“summarize”, “explain”, “compare”), and to respond in helpful, complete sentences.

In [27]:
import platform
import psutil
import torch

def detect_hardware():
    # Operating System
    system = platform.system()
    print(f"OS: {system} {platform.release()}")
    print(f"Python: {platform.python_version()}")

    # CPU
    print(f"\nCPU: {platform.processor()}")
    print(f"Cores: {psutil.cpu_count(logical=False)} physical, {psutil.cpu_count()} logical")

    # Memory
    ram = psutil.virtual_memory().total / (1024**3)
    available_ram = psutil.virtual_memory().available / (1024**3)
    print(f"\nRAM: {ram:.1f} GB total")
    print(f"Available: {available_ram:.1f} GB")

    # Check for acceleration
    print("\nAcceleration:")

    # Apple Silicon (MPS)
    if torch.backends.mps.is_available():
        print("Apple Silicon detected (MPS acceleration available)")
        device = "mps"
    # NVIDIA GPU
    elif torch.cuda.is_available():
        print(f"NVIDIA GPU detected: {torch.cuda.get_device_name(0)}")
        device = "cuda"
    # CPU only
    else:
        print("ℹUsing CPU (no GPU acceleration detected)")
        device = "cpu"

    return device

# Detect hardware
DEVICE = detect_hardware()


OS: Linux 6.6.105+
Python: 3.12.12

CPU: x86_64
Cores: 4 physical, 8 logical

RAM: 51.0 GB total
Available: 40.0 GB

Acceleration:
NVIDIA GPU detected: Tesla T4


In [28]:
DEVICE

'cuda'

We will use `Qwen2.5-1.5B-Instruct`, a lightweight 1.5-billion-parameter instruction-tuned language model from Alibaba’s Qwen 2.5 family, optimized for fast, efficient reasoning and chat on limited hardware like Google Colab GPUs or laptops.

In [33]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Alternative models by RAM requirement:
# 2-3GB RAM:  "Qwen/Qwen3Guard-Gen-0.6B" (0.6B) - Efficient safety classifier / guardrail model
# 3-4GB RAM:  "TinyLlama-1.1B" (1.1B) - Fast, good for basic tasks
# 6-7GB RAM:  "microsoft/phi-2" (2.7B) - Strong reasoning
# 7-8GB RAM:  "Qwen/Qwen3-4B" (4B) - Latest Qwen, thinking mode
# 8-9GB RAM:  "01-ai/Yi-6B" (6B) - Excellent for code
# 9-10GB RAM: "Qwen/Qwen2.5-7B" (7B) - Stable performance

print(f"Loading {MODEL_NAME}...")

try:
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # Load model with appropriate settings
    if DEVICE == "mps":  # Apple Silicon
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            trust_remote_code=True
        ).to(DEVICE)
    elif DEVICE == "cuda":  # NVIDIA GPU
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )
    else:  # CPU
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float32,  # Full precision for CPU
            low_cpu_mem_usage=True,
            trust_remote_code=True
        )

    # Set pad token if needed
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"✅ Model loaded successfully!")
    print(f"   Device: {DEVICE}")
    print(f"   Model size: ~{sum(p.numel() for p in model.parameters())/1e9:.1f}B parameters")

except Exception as e:
    print(f"❌ Error loading model: {e}")

Loading Qwen/Qwen2.5-1.5B-Instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Model loaded successfully!
   Device: cuda
   Model size: ~1.5B parameters


<a id='generation'></a>

# Text Generation

In this section, we'll define a helper function to generate text.

This process is called **inference** — it’s when we give the model an input prompt and let it predict the next words, one token at a time.

The function we’ll create will handle three main tasks:

1. **Formatting the input** — turning a text prompt into tokens the model understands.  
2. **Generating new tokens** — asking the model to produce text based on the prompt.  
3. **Decoding the output** — converting tokens back into readable text.

In [34]:
def generate_text(prompt, max_new_tokens=100, temperature=0.7):
    """Generate a clean response."""
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": prompt},
    ]

    # Convert messages into a model-friendly text format using the tokenizer's chat template.
    # `add_generation_prompt=True` tells the tokenizer to add the "assistant" prefix automatically.
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Convert the formatted text into input tensors (numerical form the model can process)
    # and move them to the correct device (CPU or GPU).
    inputs = tokenizer(formatted, return_tensors="pt").to(DEVICE)

    # Turn off gradient tracking since we're not training the model (only generating text).
    with torch.no_grad():
        output = model.generate(
            **inputs,                             # Feed input tensors into the model
            max_new_tokens=max_new_tokens,        # Limit output length
            temperature=temperature,              # Control creativity / variability
            do_sample=temperature > 0,            # Enable sampling only if temperature > 0
            top_p=0.7,                            # Nucleus sampling: model only considers top 70% likely tokens
            repetition_penalty=1.1,               # Slightly penalize repeating words
            pad_token_id=tokenizer.pad_token_id,  # Avoid padding issues
        )

    # Decode only the newly generated tokens (skip the input prompt portion)
    new_tokens = output[0][inputs["input_ids"].shape[-1]:]

    # Convert the generated tokens back to readable text, skipping special tokens like <EOS>.
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Return the clean, final string
    return response

## Test Generation

Let's generate a text using our local model.

In [35]:
prompt = "What are the main benefits of renewable energy?"

print(f"Model: {MODEL_NAME}")
print(f"Prompt: {prompt}\n")
print("Generating response...")

import time
start = time.time()
response = generate_text(prompt, max_new_tokens=150, temperature=0.7)
elapsed = time.time() - start

print("\nResponse:")
print("-" * 50)
print(response)
print("-" * 50)
print(f"\n⏱️ Generation time: {elapsed:.2f}s  (~{150/elapsed:.1f} tok/s)")

Model: Qwen/Qwen2.5-1.5B-Instruct
Prompt: What are the main benefits of renewable energy?

Generating response...

Response:
--------------------------------------------------
Renewable energy has several key benefits, including:

1. Environmental sustainability: Renewable energy sources such as solar, wind, and hydro power do not emit greenhouse gases or pollutants that contribute to air pollution, climate change, and other environmental problems.

2. Energy security: By reducing reliance on fossil fuels, renewable energy can help countries become less vulnerable to price fluctuations in oil and gas markets, which can have negative impacts on their economies.

3. Cost savings: While initial costs for some renewable technologies may be higher than traditional energy sources, they tend to be more cost-effective over time due to lower fuel costs and reduced maintenance expenses.

4. Job creation: The renewable energy sector is growing rapidly worldwide, creating new job opportunities in 

## Multiple Prompts

We can prompt using a simple loop to get a bunch of responses:

In [36]:
# Process multiple prompts efficiently
prompts = [
    "Explain machine learning in simple terms",
    "What are the causes of climate change?",
    "How does social media affect society?"
]

for i, prompt in enumerate(prompts, 1):
    print(f"Prompt {i}: {prompt}")
    response = generate_text(prompt, max_new_tokens=100, temperature=0.7)
    print(f"Response: {response}\n")
    print("-" * 50)

Prompt 1: Explain machine learning in simple terms
Response: Machine learning is a type of artificial intelligence that allows computers to learn from data without being explicitly programmed. It involves using algorithms and statistical models to analyze patterns and make predictions or decisions based on the data.

In simpler terms, imagine you have a large set of data about customers' purchase history, such as what products they buy and when they buy them. Machine learning can help identify trends and make predictions about future customer behavior by analyzing this data.

For example, if a customer has shown a pattern of buying

--------------------------------------------------
Prompt 2: What are the causes of climate change?
Response: Climate change is caused by various factors, including:

1. Greenhouse gas emissions: The burning of fossil fuels such as coal, oil and natural gas releases large amounts of carbon dioxide (CO2) into the atmosphere. This greenhouse gas traps heat fr

## Different Temperature Settings

Theoretical Max: No hard limit! You can set temperature to 10, 100, or even 1000.

Practical Max: Usually 1.5-2.0 is the useful limit.

In [37]:
# Compare different temperature settings
prompt = "Complete this sentence: 'Happiness is like a"
temperatures = [0.3, 1.0, 2.0, 5.0]

print(f"Testing temperature effects\n")
print(f"Prompt: {prompt}\n")

for temp in temperatures:
    print(f"Temperature {temp}:")
    response = generate_text(prompt, max_new_tokens=80, temperature=temp)
    print(f"{response}\n")

Testing temperature effects

Prompt: Complete this sentence: 'Happiness is like a

Temperature 0.3:
compliment, it's something that you can't see or touch but it makes everything around you feel better.'

Temperature 1.0:
Happiness is like a diamond, both rare and beautiful when you have it, but often overlooked until it's gone.'

Temperature 2.0:
Happiness is like a garden that thrives on attention, care, and tending.

To expand on this analogy:

Just as flowers require water, sunlight, soil rich in nutrients, proper drainage systems, and diligent care to grow into vibrant blooms of color and fragrance, we too need nurturing to flourish. 

Like gardening enthusiasts who take pride in the meticulous task of cultivating their plots to create the perfect

Temperature 5.0:
crumb, easily overlooked but incredibly fulfilling if you manage it. Just like that piece left accidentally on top of someone new's desk, once found can make an afternoon unforgettable.'

In both situations the crumb re

Smaller models have less diverse "creativity" - they've learned fewer patterns, so they default to common metaphors.

### Temp = 0?

When temperature = 0, the model stops sampling from a probability distribution and instead always picks the single most likely next token — this is called greedy decoding.

That means the output is completely deterministic: the same prompt will always produce the exact same response.
It’s useful when you want precise, repeatable answers (e.g., for testing or structured output), but it removes creativity and variation that come from randomness at higher temperatures.

In [38]:
# Define a simple prompt
prompt = "Explain the difference between supervised and unsupervised learning."

# Tokenize input
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

# Function to generate deterministic output
def greedy_generate(inputs):
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,       # Greedy decoding (no randomness)
            pad_token_id=tokenizer.pad_token_id
        )
    # Decode only the new text
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

# Run twice
output_1 = greedy_generate(inputs)
output_2 = greedy_generate(inputs)

print("Run 1:\n")
print(output_1)
print("\n" + "-" * 80 + "\n")
print("Run 2:\n")
print(output_2)

Run 1:

Supervised learning is a type of machine learning where the algorithm learns from labeled data, meaning that it has access to input-output pairs with known outputs. The goal of supervised learning is to learn a function that maps inputs to outputs based on the training data provided. Once the model is trained, it can be used to make predictions for new, unseen data.

Unsupervised learning, on the other hand, is a type of machine learning where the algorithm learns from unlabeled data, meaning that there are no known output labels or targets. The goal of unsupervised learning is to find patterns or structure

--------------------------------------------------------------------------------

Run 2:

Supervised learning is a type of machine learning where the algorithm learns from labeled data, meaning that it has access to input-output pairs with known outputs. The goal of supervised learning is to learn a function that maps inputs to outputs based on the training data provided. O

## Explore Probabilities

In [39]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

def get_token_probabilities(prompt, temperature=1.0):
    """Get probability distribution for next token"""

    # Tokenize - returns PyTorch tensors
    inputs = tokenizer(prompt, return_tensors="pt")

    # move to GPU/Apple Silicon if available
    if DEVICE != "cpu":
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    # Get model output (raw logits)
    with torch.no_grad():                   # Don't calculate gradients (save memory)
        outputs = model(**inputs)           # Run the model forward pass
        logits = outputs.logits[0, -1, :]   # Last token's predictions. 0 = batch item, -1 = last position in seq (after "a"), : = all vocab tokens

    # Apply temperature - this is what a model does internally
    logits_with_temp = logits / temperature

    # Convert to probabilities with softmax
    probs = F.softmax(logits_with_temp, dim=-1)

    # Get top tokens
    top_k = 20
    top_probs, top_indices = torch.topk(probs, top_k)

    # Decode tokens back to text
    tokens = [tokenizer.decode([idx.item()]) for idx in top_indices]

    return tokens, top_probs.cpu().numpy(), logits.cpu().numpy()

# Analyze a prompt
prompt = "Complete this sentence: 'Happiness is like a"
tokens, probs, raw_logits = get_token_probabilities(prompt, temperature=1.0)

# Display results
print(f"Top 20 token probabilities for: '{prompt}'\\n")
for token, prob in zip(tokens[:10], probs[:10]):
    bar = "█" * int(prob * 100)
    print(f"{token:15s} {prob:.4f} {bar}")

Top 20 token probabilities for: 'Complete this sentence: 'Happiness is like a'\n
 flower         0.1610 ████████████████
 fruit          0.1089 ██████████
 seed           0.0848 ████████
 ______         0.0548 █████
 __             0.0461 ████
 balloon        0.0317 ███
 butterfly      0.0303 ███
 garden         0.0280 ██
 ____           0.0157 █
 ___            0.0157 █


## Using Templated prompts

In [40]:
topics = ["remote work", "artificial intelligence", "meditation"]

for topic in topics:
    prompt = f"Write a brief summary about {topic}:"
    print(f"\nSummary for {topic}:")
    output = generate_text(prompt, max_new_tokens=100, temperature=0.5)
    print(output)



Summary for remote work:
Remote work, also known as telecommuting or working from home, refers to the practice of employees performing their job duties outside of an office setting and instead at home or another location. Remote workers can be assigned to different types of jobs, including software developers, marketers, salespeople, customer service representatives, and more. The benefits of remote work include increased flexibility for employees, reduced commuting time and costs, improved work-life balance, and access to a wider pool of talent. However, there may also

Summary for artificial intelligence:
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. It involves creating algorithms and models that can analyze, interpret, and make decisions based on data inputs. AI systems use machine learning techniques such as deep learning, neural networks, and natural language processing to improve their

# Using LLMs for Research

Large Language Models (LLMs) can support computational social science by helping researchers interpret, classify, or summarize complex text data at scale.

In [41]:
%pwd

'/content'

In [42]:
from google.colab import drive
import pandas as pd

# Mount your Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [43]:
df = pd.read_csv('/content/drive/MyDrive/aita_data/aita_top_subs.csv')

In [44]:
df.head()

,idint,idstr,created,self,nsfw,author,title,url,selftext,score,...,num_comments,flair_text,flair_css_class,augmented_at,augmented_count,created_date,year,month,day_of_week,text_length
0,797709732,t3_d6xoro,1568998300,1.0,0.0,DarthCharizard,META: This sub is moving towards a value syste...,NaN,I’ve enjoyed reading and posting on this sub f...,80915.0,...,6215.0,META,NaN,NaN,NaN,2019-09-20 16:51:40,2019,9,Friday,3266.0
1,1472895100,t3_ocx94s,1625315782,1.0,0.0,OnlyInQuebec9,AITA for telling my wife the lock on my daught...,NaN,My brother in-law (Sammy) lost his home shortl...,80334.0,...,5318.0,Not the A-hole,not,NaN,NaN,2021-07-03 12:36:22,2021,7,Saturday,2664.0
2,664921441,t3_azvko1,1552322462,1.0,0.0,Renegadesrule33,"UPDATE, AITA for despising my mentally handica...",NaN,"I'm back like I said I would be,. My [original...",72776.0,...,1989.0,UPDATE,NaN,NaN,NaN,2019-03-11 16:41:02,2019,3,Monday,5437.0
3,855862814,t3_e5k3z2,1575392873,1.0,0.0,throwRA-fhfsveyary,AITA for pretending to get fired when customer...,NaN,I am a high schooler with a weekend job at a c...,63526.0,...,3645.0,Not the A-hole,not,NaN,NaN,2019-12-03 17:07:53,2019,12,Tuesday,2096.0
4,756636047,t3_cihc3z,1564233111,1.0,0.0,Thunderbear998,AITA for telling my extended family how many m...,NaN,We had a family dinner this evening. My family...,54132.0,...,5190.0,Everyone Sucks,ass,NaN,NaN,2019-07-27 13:11:51,2019,7,Saturday,1662.0


In [45]:
import random

# Sample a few posts to analyze
sample_texts = random.sample(df["selftext"].dropna().tolist(), 3)

# Define a reasoning-style prompt
prompt_template = """
        You are a tool for evaluating moral dilemmas.

        Please evaluate the following post from the subreddit "Am I the Asshole".

        <dilemma>
        {text}
        </dilemma>

        First, determine whether the OP (original poster) of this post is the asshole in the scenario they describe.
        Provide a categorical label indicating your judgment of the scenario, from one of these choices:

        - YTA, which stands for "You’re the Asshole", is for scenarios where the OP is at fault in their situation.
        - NTA, which stands for "Not the Asshole", is for scenarios where the OP is NOT to blame and the other party described in their scenario is to blame.
        - ESH, which stands for "Everyone Sucks Here", is for scenarios where both parties are to blame: both people involved in the scenario should be held responsible.
        - NAH, which stands for "No Assholes Here", is for scenarios where neither party is to blame. All parties actions are justified. Nobody needs to be held accountable. Shit happens.
        - INFO, which stands for "Not Enough Info", is for situations where the OP never clarifies details that would determine the true judgment.

        Then, please provide an explanation for why you chose this label. Restrict your explanation to ONE paragraph.

"""

# Run inference on a few samples
for i, text in enumerate(sample_texts, 1):
    print(f"\nExample {i}")
    prompt = prompt_template.format(text=text[:1000])  # truncate to avoid token limits
    response = generate_text(prompt, max_new_tokens=200, temperature=0.7)
    print("-" * 50)
    print(response)


Example 1
--------------------------------------------------
**Label:** **YTA**

**Explanation:** The original poster (OP) appears to be the "asshole" in this scenario because they failed to consider the impact of their own behavior on their new neighbors' lives. While the OP initially welcomed the newcomer by bringing them donuts upon moving in, their subsequent noise levels during the night disrupted the children's sleep and caused concern among the neighbors. Additionally, the OP’s failure to address the noise issue properly demonstrates a lack of empathy and understanding towards their new neighbors’ well-being. Therefore, based on the given information, the OP can be considered the "asshole" due to their inability to accommodate the needs of the new tenant effectively.

Example 2
--------------------------------------------------
**Label:** **NTA**

**Explanation:** The OP does not appear to be the sole cause of the conflict. While the OP’s wife may have been less attentive durin

## Using Structured Output (JSON)

By prompting models to return structured JSON outputs that follow a fixed schema (validated with tools like Pydantic), we can transform qualitative social media data—like moral reasoning in r/AmItheAsshole posts—into analyzable, reproducible datasets.

In [46]:
import pandas as pd
import json
import random

# Reusable JSON instruction string
JSON_INSTRUCTIONS = {
    "aita": """
    Your response must be a single JSON object with exactly two keys: "judgment" and "explanation".
    {
    "judgment": "YTA | NTA | ESH | NAH | INFO",
    "explanation": "A clear explanation of why you chose this judgment"
    }
    Do not include any additional text, markdown formatting, or commentary.
    """
}

def analyze_aita_post(text):
    prompt = f"""
    You are analyzing moral judgments in Reddit posts from r/AmItheAsshole (AITA).
    Read the post below and decide who is at fault.
    Follow these exact instructions:
{JSON_INSTRUCTIONS['aita']}

Post:
"{text}"
"""
    response = generate_text(prompt, temperature=0.5, max_new_tokens=300)

    # Clean up code fences
    cleaned = (
        response.strip()
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )

    try:
        data = json.loads(cleaned)
        return data
    except:
        # Fallback if model output is messy
        return {"judgment": "INFO", "explanation": cleaned[:200]}

In [47]:
# Run the analysis
post = sample_texts[1]
result = analyze_aita_post(post)

# Display
print("Full AITA Post:\n")
print(post.strip())
print("\n--------------------------------------------------")
print("Model Output:")
print(json.dumps(result, indent=2))

Full AITA Post:

Context, our son is barely 1(birthday last week). I came home from work and my wife was just getting back from a run, at which point I noticed our son was in the back room sleeping. When I asked her about it, she got defensive and said "it was only a mile, and I never got far away from the house". I don't think this is safe, so i got mad at her, and the whole thing escalated into a fight where i told her what she did wasn't safe. Now shes mad at me, thinks i'm being a judgmental ass, and doesn't want to talk. She thinks it's no different than being outside in the backyard while hes napping. So whats the deal? AITA?

Edit: So I’m getting a lot of questions and concerns, so I will try and address some of them hear. Some things I have already addressed in comments. We do have a running stroller, and a treadmill. She used to be big into running, and has lost about 100 pounds, so she is starting/wanting to go again. I am incredibly supportive of her working out, but never t

---

## Stretch Goals

With Hugging Face’s transformers library, you can try out a variety of pretrained and fine-tuned models. If you finish early, explore some of these challenges:

1. Sentiment Analysis → Analyze the sentiment of AITA posts. (Hint: distilbert-base-uncased-finetuned-sst-2-english)
2. Text Classification → Classify Reddit posts by topic or category. (Hint: search Hugging Face for “text classification”)
3. Question Answering → Ask questions about an AITA post and see if the model can extract an answer. (Hint: deepset/roberta-base-squad2)
4. Summarization → Generate concise summaries of posts. (Hint: facebook/bart-large-cnn)
5. Translation → Try translating posts into another language. (Hint: Helsinki-NLP opus-mt models)